# Milestone 5 — Transform Pipeline

This notebook verifies the data pre-processing and scaling pipeline `TransformPipeline` when applied to the PyTorch dataset sample.

## 1. Load BiohubTorchDataset & TransformPipeline

In [1]:
import sys
from pathlib import Path
import torch

# Add project root to sys.path to allow importing src
project_root = str(Path("..").resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.datasets.torch_dataset import BiohubTorchDataset
from src.preprocessing.transforms import TransformPipeline
print("Modules imported successfully.")

Modules imported successfully.


## 2. Load Dataset

In [2]:
# Find matching dataset pair programmatically
train_dir = Path("../data/raw/train")
zarr_paths = sorted(list(train_dir.glob("*.zarr")))
geff_paths = sorted(list(train_dir.glob("*.geff")))

test_zarr_path = None
test_geff_path = None

for z_path in zarr_paths:
    g_path = z_path.with_suffix(".geff")
    if g_path in geff_paths:
        test_zarr_path = z_path
        test_geff_path = g_path
        break

if not test_zarr_path or not test_geff_path:
    raise FileNotFoundError("Matching Zarr/GEFF pair not found")

dataset = BiohubTorchDataset(test_zarr_path, test_geff_path)
pipeline = TransformPipeline()

2026-07-08 21:25:42,643 - INFO - Opening Zarr dataset group at: E:\biohub-cell-tracking\data\raw\train\44b6_0113de3b.zarr


2026-07-08 21:25:42,652 - INFO - Successfully opened dataset '0' with shape (100, 64, 256, 256) and dtype uint16


2026-07-08 21:25:42,653 - INFO - Loading .geff dataset at: E:\biohub-cell-tracking\data\raw\train\44b6_0113de3b.geff


2026-07-08 21:25:42,665 - INFO - Successfully loaded .geff dataset with 52 nodes and 50 edges.


## 3. Apply Transform to Sample 0

In [3]:
raw_sample = dataset[0]
transformed_sample = pipeline(raw_sample)

## 4. Print Image Properties

In [4]:
image_raw = raw_sample["image"]
image_transformed = transformed_sample["image"]

print(f"Raw Image Dtype:         {image_raw.dtype}")
# Cast raw image to float32 before computing min/max to avoid NotImplementedError on CPU for uint16
print(f"Raw Image Min/Max:       {image_raw.to(torch.float32).min()} / {image_raw.to(torch.float32).max()}")
print(f"Transformed Image Dtype: {image_transformed.dtype}")
print(f"Transformed Image Min:   {image_transformed.min()}")
print(f"Transformed Image Max:   {image_transformed.max()}")

Raw Image Dtype:         torch.uint16
Raw Image Min/Max:       12.0 / 2950.0
Transformed Image Dtype: torch.float32
Transformed Image Min:   0.00018310826271772385
Transformed Image Max:   0.04501411318778992


## 5. Verify Target Labels Unchanged

In [5]:
node_ids_unchanged = torch.equal(raw_sample["node_ids"], transformed_sample["node_ids"])
positions_unchanged = torch.equal(raw_sample["positions"], transformed_sample["positions"])
edges_unchanged = torch.equal(raw_sample["edges"], transformed_sample["edges"])

print(f"node_ids unchanged:  {node_ids_unchanged}")
print(f"positions unchanged: {positions_unchanged}")
print(f"edges unchanged:     {edges_unchanged}")

node_ids unchanged:  True
positions unchanged: True
edges unchanged:     True
